In [2]:

import re
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,
           'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,'risorse_russia':5,
           'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,
           'stabilita_russia':5,'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,
           'stabilita_rotte_cina':5,'stabilita_cina':5,'risorse_europa':5,'influenza_diplomatica_europa':5,
           'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}

FAZ_TRACKS = {
    'Iran': ['risorse_iran','forze_militari_iran','tecnologia_nucleare_iran','stabilita_iran'],
    'Coalizione': ['risorse_coalizione','influenza_diplomatica_coalizione','tecnologia_avanzata_coalizione','supporto_pubblico_coalizione','stabilita_coalizione'],
    'Russia': ['risorse_russia','influenza_militare_russia','veto_onu_russia','stabilita_economica_russia','stabilita_russia'],
    'Cina': ['risorse_cina','influenza_commerciale_cina','cyber_warfare_cina','stabilita_rotte_cina','stabilita_cina'],
    'Europa': ['risorse_europa','influenza_diplomatica_europa','aiuti_umanitari_europa','coesione_ue_europa','stabilita_europa'],
}
ALL_T = list(DEFAULT.keys())

card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)

track_pos = defaultdict(int)
track_neg = defaultdict(int)

for m in card_re.finditer(src):
    cid, cname, faz, ctype, eff_str = m.groups()
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            try:
                val_expr = fn.group(1).strip().rstrip(',')
                v = eval(f"(lambda v:{val_expr})(DEFAULT['{t}'])", {'DEFAULT':DEFAULT})
                if float(v) > 0: track_pos[t] += 1
                elif float(v) < 0: track_neg[t] += 1
            except: pass

print("=== ANALISI BIDIREZIONALITÀ (PRIMA DEL FIX) ===")
for t in ALL_T:
    pos = track_pos[t]; neg = track_neg[t]
    sym = "✅" if pos > 0 and neg > 0 else "❌"
    print(f"  {sym} {t}: +{pos} / -{neg}")


=== ANALISI BIDIREZIONALITÀ (PRIMA DEL FIX) ===
  ✅ nucleare: +31 / -26
  ✅ sanzioni: +96 / -57
  ✅ opinione: +110 / -22
  ✅ defcon: +54 / -63
  ✅ risorse: +103 / -47
  ✅ stabilita: +94 / -32
  ❌ risorse_iran: +13 / -0
  ❌ forze_militari_iran: +117 / -0
  ❌ tecnologia_nucleare_iran: +12 / -0
  ❌ stabilita_iran: +80 / -0
  ❌ risorse_coalizione: +13 / -0
  ❌ influenza_diplomatica_coalizione: +82 / -0
  ❌ tecnologia_avanzata_coalizione: +26 / -0
  ❌ supporto_pubblico_coalizione: +47 / -0
  ❌ stabilita_coalizione: +57 / -0
  ❌ risorse_russia: +16 / -0
  ❌ influenza_militare_russia: +51 / -0
  ❌ veto_onu_russia: +23 / -0
  ❌ stabilita_economica_russia: +15 / -0
  ❌ stabilita_russia: +5 / -0
  ❌ risorse_cina: +29 / -0
  ❌ influenza_commerciale_cina: +60 / -0
  ❌ cyber_warfare_cina: +12 / -0
  ❌ stabilita_rotte_cina: +30 / -0
  ❌ stabilita_cina: +1 / -0
  ❌ risorse_europa: +27 / -0
  ❌ influenza_diplomatica_europa: +57 / -0
  ❌ aiuti_umanitari_europa: +5 / -0
  ❌ coesione_ue_europa: +99 / -0


In [5]:

import re
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

card_re = re.compile(
    r"\{\s*card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)',\s*op_points:(\d+),\s*deck_type:'([^']+)',\s*description:'([^']*)'",
    re.DOTALL
)
matches = list(card_re.finditer(src))

# Regole: (track, faz, tipo|None, delta, max_carte)
fixes_needed = {
    'tecnologia_nucleare_iran': [
        ('Coalizione', 'Militare', -1, 10), ('Coalizione', 'Diplomatico', -1, 5),
        ('Coalizione', 'Segreto', -1, 4),
        ('Europa', 'Diplomatico', -1, 5),
        ('Iran', 'Militare', 1, 0), # già molte +, skip
    ],
    'forze_militari_iran': [
        ('Coalizione', 'Militare', -1, 0),  # già coperto
    ],
    'stabilita_iran': [
        ('Coalizione', 'Economico', -1, 6), ('Coalizione', 'Segreto', -1, 3),
    ],
    'stabilita_coalizione': [
        ('Iran', 'Militare', -1, 0),  # già molti
    ],
    'risorse_iran': [
        ('Coalizione', 'Economico', -1, 6), ('Coalizione', 'Militare', -1, 3),
    ],
    'risorse_coalizione': [
        ('Iran', 'Militare', -1, 4),
        ('Coalizione', 'Economico', 1, 0),  # già coperto
    ],
    'influenza_diplomatica_coalizione': [
        # già coperto da molte carte Iran/Russia con v-1
    ],
    'tecnologia_avanzata_coalizione': [
        ('Cina', 'Segreto', -1, 5), ('Russia', 'Segreto', -1, 4),
        ('Iran', 'Segreto', -1, 3),
    ],
    'supporto_pubblico_coalizione': [
        # già coperto da molte carte Iran/Russia con v-1
    ],
    'influenza_militare_russia': [
        ('Coalizione', 'Militare', -1, 6),
    ],
    'veto_onu_russia': [
        ('Coalizione', 'Diplomatico', -1, 5), ('Europa', 'Diplomatico', -1, 4),
    ],
    'stabilita_economica_russia': [
        ('Coalizione', 'Economico', -1, 5),
    ],
    'stabilita_russia': [
        ('Coalizione', 'Economico', -1, 4), ('Russia', 'Diplomatico', 1, 4),
    ],
    'risorse_russia': [
        ('Coalizione', 'Economico', -1, 4),
    ],
    'influenza_commerciale_cina': [
        ('Coalizione', 'Economico', -1, 5), ('Europa', 'Economico', -1, 4),
    ],
    'cyber_warfare_cina': [
        ('Coalizione', 'Segreto', -1, 6),
    ],
    'stabilita_rotte_cina': [
        ('Coalizione', 'Militare', -1, 4),
    ],
    'stabilita_cina': [
        ('Coalizione', 'Economico', -1, 3), ('Cina', 'Diplomatico', 1, 4),
        ('Cina', 'Economico', 1, 4),
    ],
    'risorse_cina': [
        ('Coalizione', 'Economico', -1, 3),
    ],
    'influenza_diplomatica_europa': [
        ('Iran', 'Diplomatico', -1, 4), ('Russia', 'Media', -1, 4),
    ],
    'aiuti_umanitari_europa': [
        ('Coalizione', 'Militare', -1, 3),
    ],
    'coesione_ue_europa': [
        # già coperto con molti v-1 da Iran/Russia/Cina
    ],
    'stabilita_europa': [
        ('Iran', 'Militare', -1, 4), ('Russia', 'Media', -1, 3),
        ('Europa', 'Economico', 1, 5), ('Europa', 'Diplomatico', 1, 5),
    ],
    'risorse_europa': [
        ('Iran', 'Militare', -1, 3),
    ],
}

new_src = src
total_added = 0
added_log = defaultdict(int)

for track, rules in fixes_needed.items():
    for (target_faz, target_type, delta, max_cards) in rules:
        if max_cards == 0:
            continue
        count = 0
        for idx, m in enumerate(matches):
            if count >= max_cards:
                break
            cid, cname, faz, ctype, op, deck, desc = m.groups()
            if faz != target_faz:
                continue
            if target_type and ctype != target_type:
                continue

            card_search = f"card_id:'{cid}'"
            pos = new_src.find(card_search)
            if pos == -1:
                continue

            eff_m = re.search(r'(effects:\{)([^}]+)(\})', new_src[pos:pos+700])
            if not eff_m:
                continue

            eff_str = eff_m.group(2)
            if track + ':' in eff_str:
                continue

            sign = '+' if delta > 0 else ''
            new_eff = eff_m.group(1) + eff_m.group(2) + f', {track}:(v)=>v{sign}{delta}' + eff_m.group(3)
            abs_pos = pos + eff_m.start()
            new_src = new_src[:abs_pos] + new_eff + new_src[abs_pos + len(eff_m.group(0)):]
            count += 1
            total_added += 1
            added_log[track] += 1

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts', 'w') as f:
    f.write(new_src)

print(f"Effetti aggiunti totale: {total_added}")
for t, n in sorted(added_log.items()):
    print(f"  {t}: +{n} effetti aggiunti")


Effetti aggiunti totale: 146
  aiuti_umanitari_europa: +3 effetti aggiunti
  cyber_warfare_cina: +5 effetti aggiunti
  influenza_commerciale_cina: +9 effetti aggiunti
  influenza_diplomatica_europa: +8 effetti aggiunti
  influenza_militare_russia: +6 effetti aggiunti
  risorse_cina: +3 effetti aggiunti
  risorse_coalizione: +4 effetti aggiunti
  risorse_europa: +3 effetti aggiunti
  risorse_iran: +9 effetti aggiunti
  risorse_russia: +4 effetti aggiunti
  stabilita_cina: +11 effetti aggiunti
  stabilita_economica_russia: +5 effetti aggiunti
  stabilita_europa: +17 effetti aggiunti
  stabilita_iran: +7 effetti aggiunti
  stabilita_rotte_cina: +4 effetti aggiunti
  stabilita_russia: +8 effetti aggiunti
  tecnologia_avanzata_coalizione: +7 effetti aggiunti
  tecnologia_nucleare_iran: +24 effetti aggiunti
  veto_onu_russia: +9 effetti aggiunti


In [8]:

import re
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,
           'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,'risorse_russia':5,
           'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,
           'stabilita_russia':5,'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,
           'stabilita_rotte_cina':5,'stabilita_cina':5,'risorse_europa':5,'influenza_diplomatica_europa':5,
           'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}
ALL_T = list(DEFAULT.keys())

card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)
track_pos = defaultdict(int)
track_neg = defaultdict(int)

for m in card_re.finditer(src):
    cid, cname, faz, ctype, eff_str = m.groups()
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            try:
                val_expr = fn.group(1).strip().rstrip(',')
                v = eval(f"(lambda v:{val_expr})(DEFAULT['{t}'])", {'DEFAULT':DEFAULT})
                if float(v) > 0: track_pos[t] += 1
                elif float(v) < 0: track_neg[t] += 1
            except: pass

print("=== ANALISI BIDIREZIONALITÀ (DOPO FIX) ===")
for t in ALL_T:
    pos = track_pos[t]; neg = track_neg[t]
    sym = "✅" if pos > 0 and neg > 0 else "❌"
    print(f"  {sym} {t}: +{pos} / -{neg}")


=== ANALISI BIDIREZIONALITÀ (DOPO FIX) ===
  ✅ nucleare: +31 / -26
  ✅ sanzioni: +96 / -57
  ✅ opinione: +110 / -22
  ✅ defcon: +54 / -63
  ✅ risorse: +103 / -47
  ✅ stabilita: +94 / -32
  ❌ risorse_iran: +22 / -0
  ❌ forze_militari_iran: +117 / -0
  ❌ tecnologia_nucleare_iran: +36 / -0
  ❌ stabilita_iran: +87 / -0
  ❌ risorse_coalizione: +17 / -0
  ❌ influenza_diplomatica_coalizione: +82 / -0
  ❌ tecnologia_avanzata_coalizione: +33 / -0
  ❌ supporto_pubblico_coalizione: +47 / -0
  ❌ stabilita_coalizione: +57 / -0
  ❌ risorse_russia: +20 / -0
  ❌ influenza_militare_russia: +57 / -0
  ❌ veto_onu_russia: +32 / -0
  ❌ stabilita_economica_russia: +20 / -0
  ❌ stabilita_russia: +13 / -0
  ❌ risorse_cina: +32 / -0
  ❌ influenza_commerciale_cina: +69 / -0
  ❌ cyber_warfare_cina: +17 / -0
  ❌ stabilita_rotte_cina: +34 / -0
  ❌ stabilita_cina: +12 / -0
  ❌ risorse_europa: +30 / -0
  ❌ influenza_diplomatica_europa: +65 / -0
  ❌ aiuti_umanitari_europa: +8 / -0
  ❌ coesione_ue_europa: +99 / -0
  ❌

In [11]:

import re
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

# Il problema è che v=>v-1 con v=5 dà 4 (positivo!)
# Dobbiamo misurare il DELTA rispetto al valore base
DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,
           'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,'risorse_russia':5,
           'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,
           'stabilita_russia':5,'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,
           'stabilita_rotte_cina':5,'stabilita_cina':5,'risorse_europa':5,'influenza_diplomatica_europa':5,
           'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}
ALL_T = list(DEFAULT.keys())

card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)
track_pos = defaultdict(int)
track_neg = defaultdict(int)

for m in card_re.finditer(src):
    cid, cname, faz, ctype, eff_str = m.groups()
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            try:
                val_expr = fn.group(1).strip().rstrip(',')
                base = DEFAULT[t]
                result = eval(f"(lambda v:{val_expr})({base})", {'DEFAULT':DEFAULT})
                delta = float(result) - base  # DELTA rispetto al valore iniziale
                if delta > 0: track_pos[t] += 1
                elif delta < 0: track_neg[t] += 1
            except: pass

print("=== ANALISI BIDIREZIONALITÀ CORRETTA (delta rispetto a DEFAULT) ===")
all_ok = True
for t in ALL_T:
    pos = track_pos[t]; neg = track_neg[t]
    sym = "✅" if pos > 0 and neg > 0 else "❌"
    if pos == 0 or neg == 0:
        all_ok = False
    print(f"  {sym} {t}: +{pos} / -{neg}")

print("\nTutti bidirezionali:", all_ok)


=== ANALISI BIDIREZIONALITÀ CORRETTA (delta rispetto a DEFAULT) ===
  ❌ nucleare: +0 / -57
  ❌ sanzioni: +0 / -153
  ✅ opinione: +110 / -22
  ❌ defcon: +0 / -117
  ❌ risorse: +0 / -150
  ❌ stabilita: +0 / -126
  ❌ risorse_iran: +0 / -22
  ✅ forze_militari_iran: +89 / -28
  ✅ tecnologia_nucleare_iran: +11 / -25
  ✅ stabilita_iran: +38 / -49
  ✅ risorse_coalizione: +7 / -10
  ✅ influenza_diplomatica_coalizione: +36 / -46
  ✅ tecnologia_avanzata_coalizione: +17 / -16
  ✅ supporto_pubblico_coalizione: +35 / -12
  ✅ stabilita_coalizione: +7 / -50
  ✅ risorse_russia: +16 / -4
  ✅ influenza_militare_russia: +33 / -24
  ✅ veto_onu_russia: +23 / -9
  ✅ stabilita_economica_russia: +15 / -5
  ✅ stabilita_russia: +9 / -4
  ❌ risorse_cina: +0 / -32
  ✅ influenza_commerciale_cina: +55 / -14
  ✅ cyber_warfare_cina: +8 / -9
  ✅ stabilita_rotte_cina: +29 / -5
  ✅ stabilita_cina: +9 / -3
  ✅ risorse_europa: +9 / -21
  ✅ influenza_diplomatica_europa: +50 / -15
  ✅ aiuti_umanitari_europa: +5 / -3
  ✅ coes

In [14]:

import re
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,
           'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,'risorse_russia':5,
           'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,
           'stabilita_russia':5,'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,
           'stabilita_rotte_cina':5,'stabilita_cina':5,'risorse_europa':5,'influenza_diplomatica_europa':5,
           'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}
ALL_T = list(DEFAULT.keys())

# Verifica un campione di effetti globali per capire il pattern
card_re_simple = re.compile(r"card_id:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)
# Trova le prime 5 carte con nucleare
count = 0
for m in card_re_simple.finditer(src):
    eff_str = m.group(2)
    fn = re.search(r'nucleare\s*:\s*\(v[^)]*\)\s*=>\s*([^,}]+)', eff_str)
    if fn and count < 5:
        expr = fn.group(1).strip()
        base = DEFAULT['nucleare']
        try:
            result = eval(f"(lambda v:{expr})({base})")
            delta = result - base
            print(f"card {m.group(1)}: nucleare expr={expr!r}, result={result}, delta={delta}")
        except Exception as e:
            print(f"card {m.group(1)}: expr={expr!r}, ERROR={e}")
        count += 1


card C025: expr='v<=7?2:v<=12?1:3', ERROR=invalid syntax (<string>, line 1)
card C026: nucleare expr='1', result=1, delta=-4
card C028: nucleare expr='1', result=1, delta=-4
card C030: expr='v>=11?2:1', ERROR=invalid syntax (<string>, line 1)
card C032: expr='v<=5?3:v<=10?2:1', ERROR=invalid syntax (<string>, line 1)
